# Sprint 3 Live — Core-concept workshops
Use this instructor Colab for LS09–LS12, in a separate practice workspace. Campus Sprints 1–3 use VS Code and two terminals; Sprint 1 uses its Campus Colab only for checkpoint verification. No Campus ZIP or business assignment is needed here.

The public diagnosis example is a small vehicle for the concept. Inspect the same `service/fieldcare/` source layout used locally. The supplied `live_*.py` modules make each registration/attachment visible and run in separate owned processes; in Campus those lines go in `main.py`. Each start stops only our previous process. Never restart during a quota-renewal experiment.

Run setup once. It clones `main`, installs the lock, preserves an existing marked Live folder, and performs no provider call. `ServiceProcess` starts a loopback Uvicorn child with a bounded readiness check. A Colab `127.0.0.1` address belongs to Colab, not your laptop. No foreground Uvicorn call or public tunnel is needed.

See [local setup](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/local-development.md) and [Live handoff](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/live-handoff.md).

**Continue in a fresh runtime:** download the Live ZIP and save the notebook after each session. Upload your own ZIP through the Colab Files panel (or use its local path in Jupyter), set `LIVE_ARCHIVE` below, and fill both `RESTORED_MODELS` from your saved model-choice record. Choose a `LIVE_FOLDER` that does not exist. Setup validates the ZIP and restores your edited source, router registrations and selected evidence without running it. Review that source before starting a service. After a successful restore, clear `LIVE_ARCHIVE` before rerunning setup in the same runtime so it reuses the marked folder. Credentials and indexes are excluded: re-enter provider credentials and prepare the index only when the session requires them. A Campus checkpoint cannot be restored here.


## 1. Set up once

In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, secrets, time
import httpx
REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))
from module_b.live import prepare_live_workspace, export_live_workspace, restore_live_workspace
from module_b.runtime import ServiceProcess
# On a fresh runtime, upload your own Live ZIP using the Files panel.
LIVE_ARCHIVE = ""  # Example: "/content/sprint-3-live-123.zip"; keep empty for new practice.
LIVE_FOLDER = Path.cwd() / "fieldcare-sprint-3-live"
# When restoring, explicitly select the same approved models used in that work.
RESTORED_MODELS = {"OPENROUTER_MODEL": "", "OPENROUTER_MODEL_V2": ""}
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
if LIVE_ARCHIVE:
    if any(model not in CHAT_MODELS for model in RESTORED_MODELS.values()):
        raise ValueError("Fill both RESTORED_MODELS with your saved approved model choices before restoring.")
    PROJECT = restore_live_workspace(LIVE_ARCHIVE, LIVE_FOLDER)
else:
    PROJECT = prepare_live_workspace(REPO, LIVE_FOLDER)
CALLERS = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
caller_keys = {name: secrets.token_urlsafe(32) for name in CALLERS}
previous_models = {name: globals().get("service_env", {}).get(name) for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
service_env = {variable: caller_keys[name] for name, variable in CALLERS.items()}
service_env["FIELDCARE_WORK_DIR"] = str((PROJECT / "var").resolve())
for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2"):
    service_env[name] = (RESTORED_MODELS[name] if LIVE_ARCHIVE else
                         previous_models[name] or os.environ.get(name, DEFAULT_CHAT_MODEL))
    if service_env[name] not in CHAT_MODELS:
        raise ValueError(f"Select an approved {name}.")
print("Configured models:", {name: service_env[name] for name in RESTORED_MODELS})
if globals().get("service") is not None:
    service.stop()
service = None
supported = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
incomplete = {"question": "Which filter checks are documented?"}
print("Live workspace:", PROJECT)
print("Source main:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
print("No provider or index is needed for health, validation and clarification.")

from module_b.live import start_live_service

## LS09 — Earlier content, explicit completion
Begin with the consumer: a person can use progressive text; a program may need the completed JSON. Inspect `yield` and the event format in `stream_routes.py`. In `live_stream.py`, register streaming before collecting protected POST paths and attaching the guard. The public demo starts with two callers and 2/60; keep your own Campus policy in your local project.

In [ ]:
print((PROJECT / "service/fieldcare/stream_routes.py").read_text())
print((PROJECT / "service/fieldcare/live_stream.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_stream", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    for label, body, headers in [
        ("missing key", incomplete, {}),
        ("invalid", {}, {"X-API-Key": caller_keys["dispatch"]}),
        ("clarification", incomplete, {"X-API-Key": caller_keys["dispatch"]}),
    ]:
        response = client.post("/v1/diagnose-stream", json=body, headers=headers)
        print(label, response.status_code, response.headers.get("content-type"), response.json())

### Optional provider preparation — after the deterministic checks
The same explicit preparation as Campus: stop the service, load the private provider key, then run `python -m fieldcare.prepare_index`. Embedding the supplied synthetic documents uses provider quota. Set the switch only when you intend to do this. A healthy process alone proves neither provider access nor answer quality. In local Campus, the key belongs in private `.env`; in this instructor Colab, use Colab Secrets or the hidden prompt. Restart the relevant service cell after preparation.

In [ ]:
PREPARE_PROVIDER = False
if PREPARE_PROVIDER:
    if service is not None:
        service.stop()
    from module_b.openrouter import require_openrouter_key
    require_openrouter_key(prompt=True)
    preparation_env = {**os.environ, **service_env,
                       "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
    subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"],
                   cwd=PROJECT, env=preparation_env, check=True)
else:
    print("Provider preparation pending; deterministic checks remain available.")

### Add observation after the guard, then make fresh requests
Campus's streaming walkthrough also attaches observation so the stream can be traced. Inspect `live_observed.py`: the last middleware wraps the guard, including rejected attempts. Start this composition for fresh evidence. A prior request cannot acquire a log retroactively. This deliberate restart begins a new experiment; it proves no quota renewal.

In [ ]:
print((PROJECT / "service/fieldcare/live_observed.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_observed", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
RUN_PROVIDER = False
stream_request_id = None
buffered_request_id = None
events = []
first_delta_seconds = None
if RUN_PROVIDER:
    started = time.perf_counter()
    with httpx.Client(base_url=service.base_url, timeout=120) as client:
        with client.stream("POST", "/v1/diagnose-stream", json=supported,
                           headers={"X-API-Key": caller_keys["dispatch"]}) as response:
            stream_request_id = response.headers.get("x-request-id")
            print("Stream HTTP", response.status_code, "request", stream_request_id)
            if response.status_code == 200 and "ndjson" in response.headers.get("content-type", ""):
                for line in response.iter_lines():
                    if line:
                        event = json.loads(line)
                        events.append(event)
                        if event.get("type") == "delta" and event.get("text") and first_delta_seconds is None:
                            first_delta_seconds = time.perf_counter() - started
                        print(round(time.perf_counter()-started, 3), event)
            else:
                response.read()
                print("Non-stream result:", response.text)
        stream_elapsed_seconds = time.perf_counter() - started
        buffered_started = time.perf_counter()
        response = client.post("/v1/diagnose", json=supported,
                               headers={"X-API-Key": caller_keys["partner"]})
        buffered_elapsed_seconds = time.perf_counter() - buffered_started
        buffered_request_id = response.headers.get("x-request-id")
        print("Buffered", response.status_code, buffered_request_id, response.json())
    terminal = next((e for e in reversed(events) if e.get("type") in {"complete", "error"}), None)
    print("Stream first text / reader finished (seconds):", first_delta_seconds, stream_elapsed_seconds)
    print("Buffered complete response (seconds):", buffered_elapsed_seconds)
    print("Separate requests; these timings are observations, not a benchmark.")
    print("Terminal:", terminal, "Complete:", bool(terminal and terminal["type"] == "complete"))
    print("Usage:", next((e.get("tokens") for e in events if e.get("type") == "usage"), None))
else:
    print("Real stream and buffered AI comparison pending. No output is substituted.")

### Completion reasoning and independent comparison
A transport exception or missing terminal event leaves completion unproved even if HTTP began with 200. Inspect the client's terminal check and the route's error handling. If an actual run fails, preserve that result. For a written calibration only, consider `metadata, delta, error` and `metadata, delta, connection closed`: neither permits a completed-answer claim. These event sequences are reasoning examples, not observed provider output.

Choose a caller, predict buffered versus streamed usefulness, then repeat the same supported question against both paths with available allowance. Compare first useful content and completion; two provider calls are not a benchmark. No forced provider interruption is promised.

## LS10 — Trace a fresh request to its saved record
Explain request ID, terminal outcome, model, optional tokens and latency before inspecting. Use the actual IDs from LS09 when available, plus a deterministic clarification and missing-key rejection below. Match each ID to exactly one terminal row. Keep usage unknown when absent.

### Attach your own logging configuration
Open `PROJECT/service/fieldcare/live_logging.py` in Colab’s file editor. It imports the already secured streaming service but does not attach observation. Keep its required `LOG_FIELDS`; decide whether to add `"first_content_ms"` and explain which question it answers. Do not add prompts, headers or raw exceptions.

Add this supplied attachment at the bottom of that file and save it:

```python
app.add_middleware(
    ObservationMiddleware,
    path=work_dir() / "requests.jsonl",
    routes=protected_post_paths(app),
    fields=LOG_FIELDS,
    approved_models=(openrouter_model(), openrouter_model_v2()),
)
```

This wraps the existing guard, so rejections are observed too. `configured_model` records the selected model when generation runs; `model` records an approved provider-reported identity, independently of optional token usage. These are different claims. Early rejections and clarifications have no model call.


In [ ]:
print(PROJECT / "service/fieldcare/live_logging.py")
print((PROJECT / "service/fieldcare/live_logging.py").read_text())


In [ ]:
# Start your edited composition; a fresh process loads the saved field selection.
service = start_live_service(PROJECT, "live_logging", service_env, service)
print("Own logging composition:", service.base_url)


In [ ]:
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    response = client.post("/v1/diagnose", json=incomplete)
    rejected_request_id = response.headers.get("x-request-id")
    print("Rejected", response.status_code, rejected_request_id)
    response = client.post("/v1/diagnose", json=incomplete,
                           headers={"X-API-Key": caller_keys["partner"]})
    clarification_request_id = response.headers.get("x-request-id")
    print("Admitted attempt", response.status_code, clarification_request_id)

In [ ]:
from module_b.observability import FIELDS
import ast
LOG_PATH = PROJECT / "var" / "requests.jsonl"
assert LOG_PATH.exists(), "Attach observation in live_logging.py, save and restart."
records = [json.loads(line) for line in LOG_PATH.read_text().splitlines() if line]
# Read the literal field tuple without importing or starting another service.
configuration = ast.parse((PROJECT / "service/fieldcare/live_logging.py").read_text())
selected_fields = ast.literal_eval(next(
    node.value for node in configuration.body if isinstance(node, ast.Assign)
    and any(isinstance(target, ast.Name) and target.id == "LOG_FIELDS" for target in node.targets)
))
request_ids = [rejected_request_id, clarification_request_id]
for request_id in request_ids:
    assert request_id, "This response has no observation ID; inspect the attachment."
    matches = [row for row in records if row["request_id"] == request_id]
    assert len(matches) == 1, "Find exactly this fresh request, not an older LS09 row."
    assert set(matches[0]) == set(selected_fields), "Saved fields differ from your configuration."
    print("Fresh saved record:", matches[0])
print("Configured fields verified:", selected_fields)
# If provider preparation is complete, rerun code-05 on this edited service.
# Correlate its new IDs separately; unavailable provider evidence stays pending.


### Minimize locally and check usefulness
This synthetic marker goes only into a local candidate dictionary. It is never sent to the service/provider and does not test every logging surface. The actual log is reviewed separately. Explain what remains useful after minimization; an empty record would also hide the marker.

In [ ]:
from module_b.observability import safe_record
marker = "SYNTHETIC-NOT-A-CREDENTIAL"
observed = next(row for row in records if row["request_id"] == rejected_request_id)
candidate = {**observed, "prompt": marker, "headers": {"X-API-Key": marker},
             "_approved_models": (service_env["OPENROUTER_MODEL"],)}
minimized = safe_record(candidate, fields=selected_fields, routes=("/v1/diagnose", "/v1/diagnose-stream"))
assert marker not in json.dumps(minimized)
assert minimized["request_id"] == observed["request_id"]
assert minimized["outcome"] == observed["outcome"]
print("Local minimization result:", minimized)
print("Actual saved log has prompt/header fields:", any("prompt" in r or "headers" in r for r in records))

## LS11 — A signal suggests an evaluation question
Use a real observation to choose a hypothesis. Inspect the original cases and expected documents/tools/flags before running. The guided pair is 005 (missing context) and 003 (equipment evidence). The original evaluator reads `service/data/pipeline_design.json`; the live OpenRouter graph does not use that design. A shared case or source revision does not make it a live-answer scorer.

In [ ]:
from module_b._module_a import project as original
from module_b.evaluation import load_pipeline, evaluate_selected
DESIGN_PATH = PROJECT / "service/data/pipeline_design.json"
CASE_IDS = ["EVAL-FC-005", "EVAL-FC-003"]  # Independent task: select and justify your own pair.
pipeline = load_pipeline(DESIGN_PATH)
for case_id in CASE_IDS:
    print("Original criterion:", original.eval_case_by_id(pipeline.env, case_id))
report = evaluate_selected(DESIGN_PATH, CASE_IDS)
for row in report["rows"]:
    print(row)
print("Reference-design hash:", report["design_revision"])
print("Deterministic reference evaluation; no claim about live wording or transport.")

### Independent investigation
Write the signal and hypothesis before choosing a different relevant original case and companion in `CASE_IDS`. Preserve the original input and criteria, then rerun. Explain a pass honestly; no defect or design edit is required. Record the design hash and what additional evidence your live-service claim needs. Never replay a private log body into evaluation.

## LS12 — Review the evidence chain
Show one actual stream and its terminal state when provider access is available, the same run's request ID and safe record, the buffered contract, and a justified original evaluation case/result. Provider checks remain pending when unavailable. Use the written incomplete-event example only as calibration. Reviewers distinguish transport evidence, local minimization and reference evaluation. Repair one gap or narrow one unsupported conclusion; keep Campus assessment source separate.

In [ ]:
if service is not None:
    service.stop()
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/live-models.json").write_text(json.dumps(
    {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}, indent=2) + "\n")
print("Save these configured model choices with your notebook:",
      {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")})
CHECKPOINT_OUT = PROJECT.parent / f"sprint-3-live-{time.time_ns()}.zip"
print("Download:", export_live_workspace(PROJECT, CHECKPOINT_OUT))

### Carry forward

Keep this Live workspace separate from Campus. Carry the secured, observable service checkpoint you intend to extend, with the source revision and evidence from the same run.